# Project 7: Customer Segmentation using Unsupervised Learning 🛍️📊

### 1. Executive Summary
In retail, hyper-personalized marketing is the key to maximizing ROI. This project leverages **Unsupervised Machine Learning (K-Means)** to discover hidden customer segments based on purchasing behavior. 

To make our insights truly stand out, we integrate **Plotly 3D Interactive Visualizations** and **Principal Component Analysis (PCA)**, translating raw, multi-dimensional data into actionable business strategies.

### 2. Technical Workflow
- **Data Engineering:** Standard Case formatting and feature selection.
- **Interactive EDA:** High-end, interactive 3D and 2D visualizations using Plotly.
- **Unsupervised Learning:** Elbow Method for optimal K-selection & K-Means Clustering.
- **Dimensionality Reduction:** PCA mapping for 2D cluster visualization.
- **Business Strategy:** Assigning actionable "Personas" to mathematical clusters.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# 1. Load Dataset
df = pd.read_csv('mall_customers.csv')

# 2. Elegant Data Formatting (Standard Case & Clean Columns)
df.rename(columns={
    'Genre': 'Gender', 
    'Annual Income (k$)': 'Annual Income', 
    'Spending Score (1-100)': 'Spending Score'
}, inplace=True)

# Format text data
df['Gender'] = df['Gender'].str.title()

print("Dataset Loaded and Elegantly Formatted!")
print(f"Shape: {df.shape}")
print(df.head())

Dataset Loaded and Elegantly Formatted!
Shape: (200, 5)
   CustomerID  Gender  Age  Annual Income  Spending Score
0           1    Male   19             15              39
1           2    Male   21             15              81
2           3  Female   20             16               6
3           4  Female   23             16              77
4           5  Female   31             17              40


## 3. High-End Exploratory Data Analysis (EDA)
We use `plotly_dark` templates to create engaging, neon-styled interactive plots. This allows stakeholders to hover over data points and dynamically explore customer demographics.

In [2]:
# 1. Gender Distribution (Neon Donut Chart)
fig1 = px.pie(df, names="Gender", title="Customer Gender Distribution", hole=0.4,
             color_discrete_sequence=["#00D2FF", "#FF007F"])
fig1.update_traces(textinfo="percent+label", textfont_size=16, marker=dict(line=dict(color="#000000", width=2)))
fig1.update_layout(template="plotly_dark", title_font=dict(size=22), font=dict(family="Arial"))
fig1.show()

# 2. Income vs Spending Score by Age (Interactive 3D Scatter)
fig2 = px.scatter_3d(df, x="Annual Income", y="Spending Score", z="Age", color="Gender",
                     hover_data=["Annual Income", "Spending Score"], opacity=0.8,
                     color_discrete_sequence=["#00D2FF", "#FF007F"],
                     title="3D View: Income, Spending, and Age by Gender")

fig2.update_traces(marker=dict(size=5, line=dict(width=1, color="#FFFFFF")))
fig2.update_layout(template="plotly_dark", title_font=dict(size=20), margin=dict(l=0, r=0, b=0, t=50))
fig2.show()

## 4. Unsupervised Learning: The Elbow Method & K-Means
To segment our customers, we focus on their purchasing power (`Annual Income`) and shopping habits (`Spending Score`). We will use the Elbow Method to find the mathematically optimal number of customer groups (K).

In [3]:
# Select Features for Clustering
X = df[['Annual Income', 'Spending Score']]

# Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Calculate WCSS
wcss =[]
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

# Interactive Elbow Method Plot
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=list(range(1, 11)), y=wcss, mode='lines+markers',
                          marker=dict(size=10, color='#FFEA00', line=dict(width=2, color='white')),
                          line=dict(color='#FF007F', width=3)))

fig3.update_layout(template="plotly_dark", title="The Elbow Method: Optimizing K",
                   xaxis_title="Number of Clusters (K)", yaxis_title="WCSS",
                   title_font=dict(size=22))
fig3.add_vline(x=5, line_width=2, line_dash="dash", line_color="#00D2FF", annotation_text="Optimal K=5")
fig3.show()

## 5. Model Execution, Personas & Dimensionality Reduction
With K=5, we assign our customers to 5 distinct clusters and map them to **Business Personas**. We will visualize these groups in a stunning 3D interactive plot, and then use **PCA (Principal Component Analysis)** to reduce dimensions for a clean 2D strategy map.

In [7]:
# 1. Train K-Means Model
kmeans = KMeans(n_clusters=5, init='k-means++', random_state=42)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# 2. Map Mathematical Clusters to Business Personas
persona_mapping = {
    0: 'Average Joes (Mid Income, Mid Spend)',
    1: 'Whales (High Income, High Spend)',
    2: 'Careful Spenders (High Income, Low Spend)',
    3: 'Impulse Buyers (Low Income, High Spend)',
    4: 'Budget Conscious (Low Income, Low Spend)'
}
df['Business Persona'] = df['Cluster'].map(persona_mapping)

# 3. Interactive 3D Cluster Visualization
custom_palette =["#FF007F", "#00D2FF", "#FFEA00", "#00FF00", "#B026FF"]

fig4 = px.scatter_3d(df, x="Annual Income", y="Spending Score", z="Age", color="Business Persona",
                     hover_data=["Gender"], opacity=0.9, color_discrete_sequence=custom_palette,
                     title="Customer Segments in 3D Space (Income, Spend, Age)")
fig4.update_traces(marker=dict(size=3, line=dict(width=1, color="white")))
fig4.update_layout(template="plotly_dark", margin=dict(l=0, r=0, b=80, t=50), legend=dict(yanchor="top", y=-0.1, xanchor="right", x=0.5, title_font=dict(size=10), font=dict(size=8)))
fig4.show()

# 4. Dimensionality Reduction using PCA
pca = PCA(n_components=2)
df_pca = pd.DataFrame(pca.fit_transform(X_scaled), columns=['PCA1', 'PCA2'])
df['PCA1'] = df_pca['PCA1']
df['PCA2'] = df_pca['PCA2']

# 5. Interactive 2D PCA Projection
fig5 = px.scatter(df, x="PCA1", y="PCA2", color="Business Persona",
                  hover_data=["Annual Income", "Spending Score"], size_max=15,
                  color_discrete_sequence=custom_palette,
                  title="PCA 2D Projection: The Strategic Customer Map")
fig5.update_traces(marker=dict(size=12, line=dict(width=1.5, color="black")))
fig5.update_layout(template="plotly_dark", title_font=dict(size=12), font=dict(size=8))
fig5.show()

## 6. Actionable Marketing Strategies 🎯
AI is only as good as the business decisions it drives. Based on our Unsupervised Learning clusters, here are the tailored marketing strategies for each segment:

**1. Whales (High Income, High Spend)**
* **Strategy:** Provide VIP services, exclusive loyalty programs, early access to premium products, and personalized luxury marketing. Treat them as brand ambassadors.

**2. Careful Spenders (High Income, Low Spend)**
* **Strategy:** High potential, low conversion. Avoid pushing cheap discounts. Market high-quality, durable, and value-driven products. Win their trust with quality assurance.

**3. Impulse Buyers (Low Income, High Spend)**
* **Strategy:** These customers love shopping but have budget constraints. Push limited-time offers, flash sales, BOGO (Buy One Get One), and highly engaging social media campaigns.

**4. Average Joes (Mid Income, Mid Spend)**
* **Strategy:** The largest and most stable segment. Send regular newsletters, standard promotional offers, and focus on building long-term brand loyalty without aggressive pushing.

**5. Budget Conscious (Low Income, Low Spend)**
* **Strategy:** Highly price-sensitive. Offer deep discounts, clearance sales, and budget-friendly product lines. Keep customer acquisition costs (CAC) minimal here.